## Random Forest Regression Problem

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.exceptions as px
import warnings

warnings.filterwarnings('ignore')

%matplotlib inline

In [2]:
df=pd.read_csv('cardekho_imputated.csv')

In [3]:
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


## Data Cleaning

### Handling the data 
1. Handle Missing Vlues
2. Handle Duplicates
3. Chek Data types
4. Understand The Data Set

In [4]:
## Cheking the Mising values
df.isnull().sum()

Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [5]:
# Drop the values
df.drop('car_name',axis=1,inplace=True)
df.drop('brand',axis=1,inplace= True)

In [6]:
df.head()

,Unnamed: 0,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [7]:
df['model'].unique()

<StringArray>
[        'Alto',        'Grand',          'i20',     'Ecosport',
      'Wagon R',          'i10',        'Venue',        'Swift',
        'Verna',       'Duster',
 ...
     'Panamera',      'Alturas',       'Altroz',           'NX',
     'Carnival',            'C',           'RX',        'Ghost',
 'Quattroporte',       'Gurkha']
Length: 120, dtype: str

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   model              15411 non-null  str    
 2   vehicle_age        15411 non-null  int64  
 3   km_driven          15411 non-null  int64  
 4   seller_type        15411 non-null  str    
 5   fuel_type          15411 non-null  str    
 6   transmission_type  15411 non-null  str    
 7   mileage            15411 non-null  float64
 8   engine             15411 non-null  int64  
 9   max_power          15411 non-null  float64
 10  seats              15411 non-null  int64  
 11  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), str(4)
memory usage: 1.4 MB


In [9]:
## Get all Numeric Feature 
num_feature = [feature for feature in df.columns if df[feature].dtype != 'int']
print('Num of numerical Features : ',len(num_feature))

## Categrical Featurs
cat_feature = [feature for feature in df.columns if df[feature].dtype == 'str']
print('Num of Categrical Features : ',len(cat_feature))

## For Discreate Feature
discreate_feature = [feature for feature in df.columns if len(df[feature].unique())<=25]
print('Num of discreate Features : ',len(discreate_feature))

# for continueous_feature
continueous_feature = [feature for feature in num_feature if feature not in discreate_feature]
print('Num of continoues Features : ',len(continueous_feature))

Num of numerical Features :  6
Num of Categrical Features :  4
Num of discreate Features :  5
Num of continoues Features :  3


In [10]:
df.head()

,Unnamed: 0,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [11]:
## Independed and depended Feature 
from sklearn.model_selection import train_test_split
X=df.drop(['selling_price'],axis=1)
y=df['selling_price']

In [12]:
X.tail()

,Unnamed: 0,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
15406,19537,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5
15407,19540,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7
15408,19541,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5
15409,19542,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7
15410,19543,City,2,13000,Dealer,Petrol,Automatic,18.00,1497,117.60,5


## Featurer Encoding and scaling

In [13]:
df['model'].unique()

<StringArray>
[        'Alto',        'Grand',          'i20',     'Ecosport',
      'Wagon R',          'i10',        'Venue',        'Swift',
        'Verna',       'Duster',
 ...
     'Panamera',      'Alturas',       'Altroz',           'NX',
     'Carnival',            'C',           'RX',        'Ghost',
 'Quattroporte',       'Gurkha']
Length: 120, dtype: str

In [14]:
df['model'].value_counts()

model
i20             906
Swift Dzire     890
Swift           781
Alto            778
City            757
               ... 
Altroz            1
C                 1
Ghost             1
Quattroporte      1
Gurkha            1
Name: count, Length: 120, dtype: int64

In [15]:
from sklearn.preprocessing import LabelEncoder
le =LabelEncoder()

In [16]:
# for labeling in the Model columns for categrical data 
X['model']=le.fit_transform(X['model'])

In [17]:
len(df['seller_type'].unique()),len(df['fuel_type'].unique()),len(df['transmission_type'].unique())

(3, 5, 2)

In [ ]:
    ## Crat coloumn transform with 3 types of transforms
num_feature = X.select_dtypes(exclude='object').columns  # numeric data
onhot_columns = ['seller_type','fuel_type','transmission_type'] # theses data Text columns

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

numereic_transform = StandardScaler()
oh_transform  = OneHotEncoder(drop='first')

preprocess = ColumnTransformer(
       [
           ('OnehotEncoder',oh_transform,onhot_columns),
            ('StandardScalar',numereic_transform,num_feature)
        ],remainder= 'passthrough' # for use as it as all cloumn print not any changes/delete abou this data
    )


In [19]:
X =preprocess.fit_transform(X)

In [20]:
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-1.738694,-1.519714,0.983562,1.247335,-0.000276,-1.324259,-1.263352,-0.403022
1,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-1.738516,-0.225693,-0.343933,-0.690016,-0.192071,-0.554718,-0.432571,-0.403022
2,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-1.738339,1.536377,1.647309,0.084924,-0.647583,-0.554718,-0.479113,-0.403022
3,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-1.738162,-1.519714,0.983562,-0.360667,0.292211,-0.936610,-0.779312,-0.403022
4,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-1.737985,-0.666211,-0.012060,-0.496281,0.735736,0.022918,-0.046502,-0.403022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.723327,1.508844,0.983562,-0.869744,0.026096,-0.767733,-0.757204,-0.403022
15407,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.723859,-0.556082,-1.339555,-0.728763,-0.527711,-0.216964,-0.220803,2.073444
15408,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.724036,0.407551,-0.012060,0.220539,0.344954,0.022918,0.068225,-0.403022
15409,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.724213,1.426247,-0.343933,72.541850,-0.887326,1.329794,0.917158,2.073444


In [21]:
## Now i will spliting the train and test
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

((12328, 15), (3083, 15))

## Model train and model selection


In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression,Ridge, Lasso
from  sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [23]:
##Create a Function to Evaluate Model
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [24]:
## Bigigng the Model training
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
   
}

for i in range (len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)

    ## Make Prediction
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    ## Evalution of dataset
    model_train_mae, model_train_rmse, model_train_r2 = evaluate_model(y_train,y_pred_train)

    model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test,y_pred_test)

    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 553850.0494
- Mean Absolute Error: 268104.1303
- R2 Score: 0.6218
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502582.0834
- Mean Absolute Error: 279686.6479
- R2 Score: 0.6645


Lasso
Model performance for Training set
- Root Mean Squared Error: 553850.0538
- Mean Absolute Error: 268101.7491
- R2 Score: 0.6218
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502581.1494
- Mean Absolute Error: 279682.7929
- R2 Score: 0.6645


Ridge
Model performance for Training set
- Root Mean Squared Error: 553850.6941
- Mean Absolute Error: 268061.4421
- R2 Score: 0.6218
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502572.3576
- Mean Absolute Error: 279625.1576
- R2 Score: 0.6645


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 335460.8145
- Mean 

In [ ]:
#Initialize few parameter for Hyperparamter tuning
knn_params = {"n_neighbors": [2, 3, 10, 20, 40, 50]}
rf_params = {"max_depth": [5, 8, 15, None, 10],
             "max_features": [5, 7, "auto", 8],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500, 1000]}


## Imputing Null values
1. Impute Median value for Age column
2. Impute Mode for Type of Contract
3. Impute Median for Duration of Pitch
4. Impute Mode for NumberofFollowup as it is Discrete feature
5. Impute Mode for PreferredPropertyStar
6. Impute Median for NumberofTrips
7. Impute Mode for NumberOfChildrenVisiting
8. Impute Median for MonthlyIncome

In [26]:
## model list for hyperperameter tuning
randomcv_model =  [
        ("KNN",KNeighborsRegressor(),knn_params),
        ("RF",RandomForestRegressor(),rf_params)
    ]

In [28]:
## Hyperparameter Tuning
from sklearn.model_selection import RandomizedSearchCV

model_param = {}
for name,model,params in randomcv_model:
    random = RandomizedSearchCV(estimator=model,
                                param_distributions=params,
                                n_iter=100,
                                cv=3,
                                verbose=2,
                                n_jobs=-1 )
    random.fit(X_train,y_train)
    model_param[name]= random.best_params_

for model_name in model_param:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_param[model_name])

Fitting 3 folds for each of 6 candidates, totalling 18 fits
[CV] END ......................................n_neighbors=2; total time=   0.2s
[CV] END ......................................n_neighbors=2; total time=   0.2s
[CV] END ......................................n_neighbors=3; total time=   0.2s
[CV] END .....................................n_neighbors=10; total time=   0.3s
[CV] END .....................................n_neighbors=10; total time=   0.3s
[CV] END ......................................n_neighbors=3; total time=   0.3s
[CV] END ......................................n_neighbors=2; total time=   0.3s
[CV] END .....................................n_neighbors=20; total time=   0.4s
[CV] END ......................................n_neighbors=3; total time=   0.3s
[CV] END .....................................n_neighbors=40; total time=   0.5s
[CV] END .....................................n_neighbors=40; total time=   0.6s
[CV] END .....................................n_n

In [29]:
## Retrain the model with parms
models = {
    "Random Forest Reegression": RandomForestRegressor(n_estimators=100,# Zyada trees
                                                       min_samples_split=2,
                                                       max_features=7,
                                                       max_depth=None,
                                                       n_jobs=-1),
    "K-Neighbors Regressor" :  KNeighborsRegressor(n_neighbors=10,n_jobs=-1)                                                   
                                                                                                                                                               
}

for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,y_train)

     # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)
    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')


Random Forest Reegression
Model performance for Training set
- Root Mean Squared Error: 124799.8448
- Mean Absolute Error: 35864.2436
- R2 Score: 0.9808
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 225341.6994
- Mean Absolute Error: 96587.1424
- R2 Score: 0.9325


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 376668.3802
- Mean Absolute Error: 111121.5769
- R2 Score: 0.8251
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 295441.3065
- Mean Absolute Error: 127369.3886
- R2 Score: 0.8840


